# Service MDP — Iteration de la valeur — Kernel Gateway

Exposition REST de l'**iteration de la valeur** (backups de Bellman + politique gloutonne) du carnet phare [RL-05 — MDP, programmation dynamique et Q-learning](../RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb) — programmation dynamique sur MDP, chapitre 14 d'AIMA.

Feuille de service de l'Epic AIMA-Gateway #19571. Le carnet pedagogique n'est pas modifie : ce dossier `gateway/` ajoute un carnet de service a cote, sur le meme motif que les pilotes `GameTheory/gateway/`, `Probas/Infer/gateway/`, `Z3-API/gateway/` et `NLP/gateway/`.

**Contrat** : `POST /rl/mdp-value-iteration` — corps = MDP (etats, actions, transitions `T`, recompenses `R`, `gamma`), reponse = politique optimale + fonction de valeur. Le detail du contrat, le gridworld 4x3 de demonstration et le verdict SOTA sont dans le [README](README.md) de ce dossier.

In [1]:
# Preparation : charger le coeur du service (testable hors kernel) et construire
# le gridworld 4x3 d'AIMA ch. 14 (2 obstacles, 2 terminaux +1 / -1, bruit 0.8/0.2).
import importlib.util
from pathlib import Path

# Le coeur vit a cote du carnet de service : le kernel est demarre dans ce dossier,
# que ce soit par kernel_gateway ou par papermill/nbconvert.
_core_path = Path("mdp_value_iteration_core.py")
_spec = importlib.util.spec_from_file_location("mdp_value_iteration_core", _core_path)
core = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(core)
run = core.run


def build_gridworld():
    """Le gridworld 4x3 d'AIMA ch. 14, en MDP complet (T normalises, R par (etat, action))."""
    cols, rows = 4, 3
    obstacles = {(1, 1), (1, 3)}
    terminal_reward = {(0, 3): 1.0, (2, 0): -1.0}
    step_cost = -0.04
    cells = [(r, c) for r in range(rows) for c in range(cols) if (r, c) not in obstacles]
    states = [f"{r},{c}" for (r, c) in cells]
    actions = ["N", "S", "E", "W"]
    delta = {"N": (-1, 0), "S": (1, 0), "E": (0, 1), "W": (0, -1)}

    def in_bounds(r, c):
        return 0 <= r < rows and 0 <= c < cols

    def is_obs(r, c):
        return (r, c) in obstacles

    T, R = {}, {}
    for (r, c) in cells:
        T[f"{r},{c}"], R[f"{r},{c}"] = {}, {}
        for a in actions:
            dr, dc = delta[a]
            nr, nc = r + dr, c + dc
            if not in_bounds(nr, nc) or is_obs(nr, nc):
                nr, nc = r, c  # heurter un mur / un obstacle -> rester sur place
            perps = [(-1, 0), (1, 0)] if a in ("N", "S") else [(0, -1), (0, 1)]
            outcomes = {}

            def add(rr, cc, p, _r=r, _c=c):
                if not in_bounds(rr, cc) or is_obs(rr, cc):
                    rr, cc = _r, _c
                key = f"{rr},{cc}"
                outcomes[key] = outcomes.get(key, 0.0) + p

            add(nr, nc, 0.8)
            add(r + perps[0][0], c + perps[0][1], 0.1)
            add(r + perps[1][0], c + perps[1][1], 0.1)
            T[f"{r},{c}"][a] = outcomes
            R[f"{r},{c}"][a] = terminal_reward.get((r, c), step_cost)
    return {"states": states, "actions": actions, "T": T, "R": R, "gamma": 0.9}


mdp = build_gridworld()
print("Coeur charge :", _core_path.name)
print("Etats       :", len(mdp["states"]), "(gridworld 4x3, 2 obstacles)")
print("Actions     :", mdp["actions"])

Coeur charge : mdp_value_iteration_core.py
Etats       : 10 (gridworld 4x3, 2 obstacles)
Actions     : ['N', 'S', 'E', 'W']


In [2]:
# POST /rl/mdp-value-iteration
#
# Point d'entree du service. En mode gateway, kernel_gateway route les POST sur
# /rl/mdp-value-iteration vers `http_server` ; le corps JSON est passe a `run()` du coeur et la
# reponse est serialisee. Le carnet s'execute aussi hors gateway
# (papermill/nbconvert) : `kernel_gateway` n'est alors pas importe.
import json

try:  # mode gateway : le type HTTPRequest existe
    from kernel_gateway import HTTPRequest  # noqa: F401
    _GATEWAY = True
except ImportError:  # mode hors gateway (papermill/nbconvert)
    _GATEWAY = False


def http_server(request, kernel_session):
    """Route `POST /rl/mdp-value-iteration` : `run()` du coeur, reponse JSON au contrat."""
    try:
        body = json.loads(request.body)
    except Exception as exc:
        payload = {"status": "invalid", "error": f"corps JSON invalide: {exc}"}
        return 200, json.dumps(payload, ensure_ascii=False).encode("utf-8")
    return 200, json.dumps(run(body), ensure_ascii=False).encode("utf-8")


print("Mode gateway :", "oui" if _GATEWAY else "non (demo hors gateway)")

Mode gateway :

non (demo hors gateway)

In [3]:
# Demonstration : iteration de la valeur sur le gridworld 4x3 (sortie reelle).
import json

response = run(mdp)
print("Iterations (backups de Bellman) :", response["iterations"])
print("Converge                        :", response["converged"])
print("Politique optimale (extrait)    :")
for s in ("0,0", "0,3", "2,0", "2,3"):
    print(f"  {s} -> {response['policy'][s]}")
print("Valeurs optimales (extrait)     :")
for s in ("0,0", "0,3", "2,0", "2,3"):
    print(f"  V({s}) = {response['value'][s]:.4f}")

Iterations (backups de Bellman) :

133

Converge                        :

True

Politique optimale (extrait)    :

  0,0 -> E

... (4 outputs left out)

  V(0,0) = 6.7708

  V(0,3) = 10.0000

  V(2,0) = 4.0735

  V(2,3) = 5.9828

In [4]:
# Demonstration : entree hors contrat -> `status: invalid`, aucune exception.
import json

bad = json.loads(json.dumps(mdp))  # copie profonde
bad["T"]["0,0"]["N"]["0,0"] = 0.5  # distribution de transition non normalisee
print(json.dumps(run(bad), ensure_ascii=False, indent=2))

{
  "status": "invalid",
  "error": "T['0,0']['N']: somme = 0.600000, distribution normalisee (somme ~ 1) attendue"
}

## Lecture du resultat

L'iteration de la valeur applique des **backups de Bellman** jusqu'a convergence : chaque etat prend la meilleure (recompense immediate + valeur escompee des etats successeurs), et la boucle s'arrete quand la variation max de la fonction de valeur passe sous `theta`. La **politique gloutonne** derivee (argmax du backup) est optimale.

Sur le gridworld 4x3, la politique optimale mene vers le terminal `+1` (coin haut-droit) et fuit le terminal `-1` (coin bas-gauche), en tenant compte du bruit 0.8/0.2 des deplacements.

**Verdict SOTA : SOTA-OK.** Le service expose l'iteration de la valeur du carnet phare — l'**organe natif** de la serie RL pour cet algorithme (AIMA ch. 14). La reprise est **auto-contenue et fidele** au carnet : meme algorithme, meme gridworld. Copie pedagogique declaree : le carnet enseigne precisement l'algorithme que le service expose ; il n'y a pas de bibliotheque externe que le service pretendrait utiliser sans la faire (la serie implémente l'iteration de la valeur a la main, elle n'adapte pas de solveur RL externe a ce cas).